# VGG 風格小 CNN

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/01-small-cnn/)

先執行環境格，再閱讀、執行下方完整實驗。各節互相獨立，不需要上一節的 runtime。

本版使用 CPU 驗證機制及短步參數更新；長訓練的辨識效果、AP 與 GPU 效率仍待補測。這些範例沒有預訓練權重，也不需下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.1.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 本輪只驗證 CPU。已裝相同 PyTorch 版本時保留其 CPU/CUDA build。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""Three CPU training steps and an honest prediction panel, not a trained benchmark."""
from pathlib import Path
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/miniyolo-matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/miniyolo-cache")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
from torch import nn


def make_batch():
    images = torch.zeros(8, 3, 32, 32)
    labels = torch.arange(8) % 2
    for i in range(8):
        left, top = 4 + i % 3 * 4, 6 + i % 2 * 5
        images[i, 0 if labels[i] == 0 else 2, top:top + 12, left:left + 12] = 1
    return images, labels


class SmallCNN(nn.Module):
    def __init__(self, width=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, width, 3, padding=1), nn.ReLU(),
            nn.Conv2d(width, width, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(width, width * 2, 3, padding=1), nn.ReLU(),
            nn.Conv2d(width * 2, width * 2, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(width * 2, 2)

    def forward(self, x):
        return self.head(self.pool(self.features(x)).flatten(1))


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    images, labels = make_batch()
    assert images.shape == (8, 3, 32, 32) and images.dtype == torch.float32
    assert images.min() >= 0 and images.max() <= 1
    model = SmallCNN()
    x = images
    macs = 0
    for layer in model.features:
        x = layer(x)
        if isinstance(layer, (nn.Conv2d, nn.MaxPool2d)):
            print(f"{type(layer).__name__}: {tuple(x.shape)}")
        if isinstance(layer, nn.Conv2d):
            macs += x[0].numel() * layer.in_channels * layer.kernel_size[0] ** 2
    macs += model.head.in_features * model.head.out_features
    params = sum(p.numel() for p in model.parameters())
    print(f"parameters={params}; multiply-accumulates/image={macs}")
    assert params == 1158 and macs == 479248
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
    before = model.features[0].weight.detach().clone()
    model.train()
    for step in range(3):
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = nn.functional.cross_entropy(logits, labels)
        loss.backward()
        assert torch.isfinite(model.features[0].weight.grad).all()
        optimizer.step()
        print(f"step={step}, loss={loss.item():.4f}")
    assert not torch.equal(before, model.features[0].weight)
    model.eval()
    with torch.no_grad():
        predictions = model(images).argmax(1)
    errors = (predictions != labels).nonzero().flatten().tolist()
    print(f"predictions={predictions.tolist()}, labels={labels.tolist()}, error_indices={errors}")
    print("Three-step smoke test only; accuracy here is not held-out performance.")
    fig, axes = plt.subplots(2, 4, figsize=(9, 6), constrained_layout=True)
    for i, ax in enumerate(axes.flat):
        ax.imshow(images[i].permute(1, 2, 0).numpy())
        ax.set_title(f"{i}: GT={labels[i].item()} pred={predictions[i].item()}",
                     color="red" if i in errors else "black")
        ax.axis("off")
    fig.suptitle("Synthetic images; only 3 updates; red titles = mistakes")
    output = Path("artifacts/01-small-cnn.png")
    output.parent.mkdir(exist_ok=True)
    fig.savefig(output, dpi=130)
    plt.close(fig)
    print(f"panel={output}")


if __name__ == "__main__":
    main()


Conv2d: (8, 4, 32, 32)
Conv2d: (8, 4, 32, 32)
MaxPool2d: (8, 4, 16, 16)
Conv2d: (8, 8, 16, 16)
Conv2d: (8, 8, 16, 16)
MaxPool2d: (8, 8, 8, 8)
parameters=1158; multiply-accumulates/image=479248
step=0, loss=0.6941
step=1, loss=0.6940
step=2, loss=0.6940
predictions=[1, 1, 1, 1, 1, 1, 1, 1], labels=[0, 1, 0, 1, 0, 1, 0, 1], error_indices=[0, 2, 4, 6]
Three-step smoke test only; accuracy here is not held-out performance.
panel=artifacts/01-small-cnn.png
